In [1]:
# !pip install -U numba statsforecast datasetsforecast
import numpy as np
import pandas as pd

#obtain hierarchical dataset
from datasetsforecast.hierarchical import HierarchicalData

# compute base forecast no coherent
from statsforecast.core import StatsForecast
from statsforecast.models import AutoARIMA, Naive

#obtain hierarchical reconciliation methods and evaluation
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.evaluation import evaluate
from hierarchicalforecast.methods import BottomUp, TopDown, MiddleOut
from utilsforecast.losses import mse

x:\Egyetem\semester_six\forecast_research\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load TourismSmall dataset
Y_df, S_df, tags = HierarchicalData.load('./data', 'TourismSmall')
Y_df['ds'] = pd.to_datetime(Y_df['ds'])
S_df = S_df.reset_index(names="unique_id")

100%|██████████| 1.30M/1.30M [00:00<00:00, 1.78MiB/s]
INFO:datasetsforecast.utils:Successfully downloaded datasets.zip, 1297274, bytes.
INFO:datasetsforecast.utils:Decompressing zip file...
INFO:datasetsforecast.utils:Successfully decompressed data\hierarchical\datasets.zip
100%|██████████| 335k/335k [00:00<00:00, 3.22MiB/s]
INFO:datasetsforecast.utils:Successfully downloaded OldTraffic.zip, 335471, bytes.
INFO:datasetsforecast.utils:Decompressing zip file...
INFO:datasetsforecast.utils:Successfully decompressed data\hierarchical\OldTraffic.zip
100%|██████████| 968k/968k [00:00<00:00, 5.72MiB/s]
INFO:datasetsforecast.utils:Successfully downloaded OldTourismLarge.zip, 967629, bytes.
INFO:datasetsforecast.utils:Decompressing zip file...
INFO:datasetsforecast.utils:Successfully decompressed data\hierarchical\OldTourismLarge.zip


In [8]:
Y_df['unique_id'].unique()

array(['total', 'hol', 'vfr', 'bus', 'oth', 'nsw-hol', 'vic-hol',
       'qld-hol', 'sa-hol', 'wa-hol', 'tas-hol', 'nt-hol', 'nsw-vfr',
       'vic-vfr', 'qld-vfr', 'sa-vfr', 'wa-vfr', 'tas-vfr', 'nt-vfr',
       'nsw-bus', 'vic-bus', 'qld-bus', 'sa-bus', 'wa-bus', 'tas-bus',
       'nt-bus', 'nsw-oth', 'vic-oth', 'qld-oth', 'sa-oth', 'wa-oth',
       'tas-oth', 'nt-oth', 'nsw-hol-city', 'nsw-hol-noncity',
       'vic-hol-city', 'vic-hol-noncity', 'qld-hol-city',
       'qld-hol-noncity', 'sa-hol-city', 'sa-hol-noncity', 'wa-hol-city',
       'wa-hol-noncity', 'tas-hol-city', 'tas-hol-noncity', 'nt-hol-city',
       'nt-hol-noncity', 'nsw-vfr-city', 'nsw-vfr-noncity',
       'vic-vfr-city', 'vic-vfr-noncity', 'qld-vfr-city',
       'qld-vfr-noncity', 'sa-vfr-city', 'sa-vfr-noncity', 'wa-vfr-city',
       'wa-vfr-noncity', 'tas-vfr-city', 'tas-vfr-noncity', 'nt-vfr-city',
       'nt-vfr-noncity', 'nsw-bus-city', 'nsw-bus-noncity',
       'vic-bus-city', 'vic-bus-noncity', 'qld-bus-city'

In [16]:
Y_df[Y_df['unique_id'] == 'total']

,unique_id,ds,y
0,total,1998-03-31,84503
1,total,1998-06-30,65312
2,total,1998-09-30,72753
3,total,1998-12-31,70880
4,total,1999-03-31,86893
5,total,1999-06-30,66866
6,total,1999-09-30,72182
7,total,1999-12-31,68318
8,total,2000-03-31,85651
9,total,2000-06-30,64467


In [10]:
S_df

,unique_id,nsw-hol-city,nsw-hol-noncity,vic-hol-city,vic-hol-noncity,qld-hol-city,qld-hol-noncity,sa-hol-city,sa-hol-noncity,wa-hol-city,...,qld-oth-city,qld-oth-noncity,sa-oth-city,sa-oth-noncity,wa-oth-city,wa-oth-noncity,tas-oth-city,tas-oth-noncity,nt-oth-city,nt-oth-noncity
0,total,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,hol,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,vfr,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,bus,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,oth,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,wa-oth-noncity,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
85,tas-oth-city,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
86,tas-oth-noncity,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
87,nt-oth-city,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [11]:
tags

{'Country': array(['total'], dtype=object),
 'Country/Purpose': array(['hol', 'vfr', 'bus', 'oth'], dtype=object),
 'Country/Purpose/State': array(['nsw-hol', 'vic-hol', 'qld-hol', 'sa-hol', 'wa-hol', 'tas-hol',
        'nt-hol', 'nsw-vfr', 'vic-vfr', 'qld-vfr', 'sa-vfr', 'wa-vfr',
        'tas-vfr', 'nt-vfr', 'nsw-bus', 'vic-bus', 'qld-bus', 'sa-bus',
        'wa-bus', 'tas-bus', 'nt-bus', 'nsw-oth', 'vic-oth', 'qld-oth',
        'sa-oth', 'wa-oth', 'tas-oth', 'nt-oth'], dtype=object),
 'Country/Purpose/State/CityNonCity': array(['nsw-hol-city', 'nsw-hol-noncity', 'vic-hol-city',
        'vic-hol-noncity', 'qld-hol-city', 'qld-hol-noncity',
        'sa-hol-city', 'sa-hol-noncity', 'wa-hol-city', 'wa-hol-noncity',
        'tas-hol-city', 'tas-hol-noncity', 'nt-hol-city', 'nt-hol-noncity',
        'nsw-vfr-city', 'nsw-vfr-noncity', 'vic-vfr-city',
        'vic-vfr-noncity', 'qld-vfr-city', 'qld-vfr-noncity',
        'sa-vfr-city', 'sa-vfr-noncity', 'wa-vfr-city', 'wa-vfr-noncity',
     

In [17]:
#split train/test sets
Y_test_df  = Y_df.groupby('unique_id').tail(4)
Y_train_df = Y_df.drop(Y_test_df.index)

In [19]:
Y_test_df

,unique_id,ds,y
32,total,2006-03-31,82637
33,total,2006-06-30,67523
34,total,2006-09-30,65938
35,total,2006-12-31,69544
68,hol,2006-03-31,43601
...,...,...,...
3167,nt-oth-city,2006-12-31,27
3200,nt-oth-noncity,2006-03-31,25
3201,nt-oth-noncity,2006-06-30,52
3202,nt-oth-noncity,2006-09-30,72


In [20]:
# Compute base auto-ARIMA predictions
fcst = StatsForecast(models=[AutoARIMA(season_length=4), Naive()],
                     freq='QE', n_jobs=-1)
Y_hat_df = fcst.forecast(df=Y_train_df, h=4)

In [22]:
# Reconcile the base predictions
reconcilers = [
    BottomUp(),
    TopDown(method='forecast_proportions'),
    MiddleOut(middle_level='Country/Purpose/State',
              top_down_method='forecast_proportions')
]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(Y_hat_df=Y_hat_df, Y_df=Y_train_df,
                          S_df=S_df, tags=tags)

In [23]:
Y_rec_df

,unique_id,ds,AutoARIMA,Naive,AutoARIMA/BottomUp,Naive/BottomUp,AutoARIMA/TopDown_method-forecast_proportions,Naive/TopDown_method-forecast_proportions,AutoARIMA/MiddleOut_middle_level-Country/Purpose/State_top_down_method-forecast_proportions,Naive/MiddleOut_middle_level-Country/Purpose/State_top_down_method-forecast_proportions
0,total,2006-03-31,85909.171875,63392.0,83511.706752,63392.0,85909.171875,63392.0,83060.420328,63392.0
1,total,2006-06-30,63981.953125,63392.0,64479.750538,63392.0,63981.953125,63392.0,63271.392647,63392.0
2,total,2006-09-30,70983.882812,63392.0,68372.945129,63392.0,70983.882812,63392.0,67682.479492,63392.0
3,total,2006-12-31,67429.898438,63392.0,66731.811958,63392.0,67429.898438,63392.0,65948.033310,63392.0
4,hol,2006-03-31,42805.347656,26418.0,42314.610321,26418.0,45029.655357,26418.0,42309.070190,26418.0
...,...,...,...,...,...,...,...,...,...,...
351,nt-oth-city,2006-12-31,103.004616,49.0,103.004616,49.0,151.886914,49.0,138.776638,49.0
352,nt-oth-noncity,2006-03-31,10.252673,59.0,10.252673,59.0,1.249987,59.0,1.190725,59.0
353,nt-oth-noncity,2006-06-30,39.563168,59.0,39.563168,59.0,35.688619,59.0,33.336497,59.0
354,nt-oth-noncity,2006-09-30,283.413483,59.0,283.413483,59.0,310.417726,59.0,283.415188,59.0


In [24]:
df = Y_rec_df.merge(Y_test_df, on=['unique_id', 'ds'])
evaluation = evaluate(df = df,
                      tags = tags,
                      metrics = [mse],
                      benchmark = "Naive")
evaluation

,level,metric,AutoARIMA,Naive,AutoARIMA/BottomUp,Naive/BottomUp,AutoARIMA/TopDown_method-forecast_proportions,Naive/TopDown_method-forecast_proportions,AutoARIMA/MiddleOut_middle_level-Country/Purpose/State_top_down_method-forecast_proportions,Naive/MiddleOut_middle_level-Country/Purpose/State_top_down_method-forecast_proportions
0,Country,mse-scaled,0.123161,1.0,0.055271,1.0,0.123161,1.0,0.079279,1.0
1,Country/Purpose,mse-scaled,0.171063,1.0,0.077691,1.0,0.101570,1.0,0.104186,1.0
2,Country/Purpose/State,mse-scaled,0.194383,1.0,0.149163,1.0,0.201738,1.0,0.194383,1.0
3,Country/Purpose/State/CityNonCity,mse-scaled,0.170373,1.0,0.170373,1.0,0.210061,1.0,0.203327,1.0
4,Overall,mse-scaled,0.154912,1.0,0.085346,1.0,0.131308,1.0,0.113637,1.0
